###1) Przygotowanie pełnego okresu TEST

In [0]:
from datetime import date, datetime, time, timezone
from zoneinfo import ZoneInfo

from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text(
    "validation_end_date_local_exclusive",
    "2024-10-01"
)
dbutils.widgets.text(
    "evaluation_end_date_local_exclusive",
    "2024-11-01"
)

TEST_START_DATE_TEXT = dbutils.widgets.get(
    "validation_end_date_local_exclusive"
)
TEST_END_DATE_TEXT = dbutils.widgets.get(
    "evaluation_end_date_local_exclusive"
)

MARKET_TIMEZONE = "America/Los_Angeles"
LOCAL_TIMEZONE = ZoneInfo(MARKET_TIMEZONE)

test_start_local = datetime.combine(
    date.fromisoformat(TEST_START_DATE_TEXT),
    time.min,
    tzinfo=LOCAL_TIMEZONE,
)
test_end_local = datetime.combine(
    date.fromisoformat(TEST_END_DATE_TEXT),
    time.min,
    tzinfo=LOCAL_TIMEZONE,
)

if test_end_local <= test_start_local:
    raise ValueError("Koniec TEST musi być późniejszy od początku.")

TEST_START_UTC = test_start_local.astimezone(timezone.utc)
TEST_END_UTC = test_end_local.astimezone(timezone.utc)

GOLD_REPLAY_TABLE = (
    "workspace.energy_gold.market_decision_replay_15m"
)
NEAR_PREDICTIONS_TABLE = (
    "workspace.energy_gold.market_model_test_predictions_15m"
)
DAY_AHEAD_SCHEDULE_TABLE = (
    "workspace.energy_gold.day_ahead_schedule_hourly"
)

gold_test_df = (
    spark.table(GOLD_REPLAY_TABLE)
    .filter(
        (F.col("target_interval_start_utc") >= F.lit(TEST_START_UTC))
        & (F.col("target_interval_start_utc") < F.lit(TEST_END_UTC))
    )
    .withColumn(
        "dam_hour_start_utc",
        F.date_trunc("hour", "target_interval_start_utc"),
    )
)

near_predictions_df = (
    spark.table(NEAR_PREDICTIONS_TABLE)
    .select(
        "decision_id",
        F.col("predicted_pv_energy_mwh").alias(
            "near_predicted_pv_energy_mwh"
        ),
        F.col("predicted_fmm_lmp_usd_mwh").alias(
            "near_predicted_fmm_lmp_usd_mwh"
        ),
    )
)

day_ahead_schedule_df = (
    spark.table(DAY_AHEAD_SCHEDULE_TABLE)
    .select(
        F.col("interval_start_utc").alias("dam_hour_start_utc"),
        "recommended_dam_schedule_mwh",
        F.col("predicted_pv_energy_mwh").alias(
            "day_ahead_predicted_pv_energy_mwh"
        ),
    )
)

backtest_input_df = (
    gold_test_df
    .join(near_predictions_df, on="decision_id", how="left")
    .join(
        day_ahead_schedule_df,
        on="dam_hour_start_utc",
        how="left",
    )
    .withColumn(
        "dam_schedule_15m_mwh",
        F.col("recommended_dam_schedule_mwh") / F.lit(4.0),
    )
    .withColumn(
        "predicted_pv_energy_15m_mwh",
        F.coalesce(
            F.col("near_predicted_pv_energy_mwh"),
            F.col("day_ahead_predicted_pv_energy_mwh") / F.lit(4.0),
        ),
    )
    .withColumn(
        "predicted_fmm_lmp_usd_mwh",
        F.coalesce(
            F.col("near_predicted_fmm_lmp_usd_mwh"),
            F.col("dam_lmp_usd_mwh"),
        ),
    )
    .withColumn(
        "prediction_source",
        F.when(
            F.col("near_predicted_pv_energy_mwh").isNotNull(),
            F.lit("NEAR_DELIVERY_MODEL"),
        ).otherwise(F.lit("DAY_AHEAD_FALLBACK")),
    )
)

display(
    backtest_input_df.agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_count"
        ),
        F.sum(
            F.col("recommended_dam_schedule_mwh").isNull().cast("int")
        ).alias("missing_dam_schedule_count"),
        F.sum(
            F.col("actual_pv_energy_mwh").isNull().cast("int")
        ).alias("missing_actual_pv_count"),
        F.sum(
            (F.col("prediction_source") == "NEAR_DELIVERY_MODEL")
            .cast("int")
        ).alias("near_prediction_count"),
        F.sum(
            (F.col("prediction_source") == "DAY_AHEAD_FALLBACK")
            .cast("int")
        ).alias("fallback_count"),
        F.min("target_interval_start_utc").alias("min_test_time_utc"),
        F.max("target_interval_end_utc").alias("max_test_time_utc"),
    )
)

###2) Parametry magazynu i progi cenowe

In [0]:
import math

MODEL_DATASET_TABLE = (
    "workspace.energy_gold."
    "market_model_dataset_15m"
)

INTERVAL_HOURS = 0.25

BATTERY_POWER_MW = 0.5
BATTERY_CAPACITY_MWH = 1.0

BATTERY_MIN_SOC_MWH = 0.1
BATTERY_MAX_SOC_MWH = 0.9
BATTERY_INITIAL_SOC_MWH = 0.5

ROUND_TRIP_EFFICIENCY = 0.90

CHARGE_EFFICIENCY = math.sqrt(
    ROUND_TRIP_EFFICIENCY
)

DISCHARGE_EFFICIENCY = math.sqrt(
    ROUND_TRIP_EFFICIENCY
)

POI_EXPORT_LIMIT_MW = 0.8
POI_IMPORT_LIMIT_MW = 0.5

DEGRADATION_COST_USD_MWH = 10.0

MAX_CHARGE_PER_INTERVAL_MWH = (
    BATTERY_POWER_MW * INTERVAL_HOURS
)

MAX_DISCHARGE_PER_INTERVAL_MWH = (
    BATTERY_POWER_MW * INTERVAL_HOURS
)

MAX_EXPORT_PER_INTERVAL_MWH = (
    POI_EXPORT_LIMIT_MW * INTERVAL_HOURS
)

MAX_IMPORT_PER_INTERVAL_MWH = (
    POI_IMPORT_LIMIT_MW * INTERVAL_HOURS
)

price_thresholds = (
    spark.table(MODEL_DATASET_TABLE)
    .filter("dataset_split = 'TRAIN'")
    .agg(
        F.percentile_approx(
            "dam_lmp_usd_mwh",
            0.25
        ).alias("low_price_threshold"),
        F.percentile_approx(
            "dam_lmp_usd_mwh",
            0.75
        ).alias("high_price_threshold")
    )
    .first()
)

LOW_PRICE_THRESHOLD = float(
    price_thresholds["low_price_threshold"]
)

HIGH_PRICE_THRESHOLD = float(
    price_thresholds["high_price_threshold"]
)

print(
    f"Magazyn: {BATTERY_POWER_MW} MW / "
    f"{BATTERY_CAPACITY_MWH} MWh"
)

print(
    f"Zakres SOC: "
    f"{BATTERY_MIN_SOC_MWH}–"
    f"{BATTERY_MAX_SOC_MWH} MWh"
)

print(
    f"Próg niskiej ceny: "
    f"{LOW_PRICE_THRESHOLD:.2f} USD/MWh"
)

print(
    f"Próg wysokiej ceny: "
    f"{HIGH_PRICE_THRESHOLD:.2f} USD/MWh"
)

###3) Symulacja trzech strategii magazynu

In [0]:
import numpy as np
import pandas as pd

backtest_pdf = (
    backtest_input_df
    .select(
        "decision_id",
        "decision_time_utc",
        "target_interval_start_utc",
        "target_interval_end_utc",
        "prediction_source",
        "dam_schedule_15m_mwh",
        "predicted_pv_energy_15m_mwh",
        "predicted_fmm_lmp_usd_mwh",
        "actual_pv_energy_mwh",
        "dam_lmp_usd_mwh",
        "actual_fmm_lmp_usd_mwh",
    )
    .orderBy("target_interval_start_utc")
    .toPandas()
)

if backtest_pdf.empty:
    raise RuntimeError("Zbiór TEST do backtestu jest pusty.")

SCENARIOS = [
    "NO_BATTERY",
    "BATTERY_BALANCING_PV_ONLY",
    "BATTERY_PRICE_GRID",
]


def simulate_scenario(scenario_name):
    soc_mwh = BATTERY_INITIAL_SOC_MWH
    rows = []

    for _, row in backtest_pdf.iterrows():
        schedule_mwh = max(float(row["dam_schedule_15m_mwh"]), 0.0)
        predicted_pv_mwh = max(
            float(row["predicted_pv_energy_15m_mwh"]),
            0.0,
        )
        actual_pv_mwh = max(float(row["actual_pv_energy_mwh"]), 0.0)
        predicted_price = float(row["predicted_fmm_lmp_usd_mwh"])
        actual_fmm_price = float(row["actual_fmm_lmp_usd_mwh"])
        dam_price = float(row["dam_lmp_usd_mwh"])

        if predicted_price <= LOW_PRICE_THRESHOLD:
            price_signal = "LOW"
        elif predicted_price >= HIGH_PRICE_THRESHOLD:
            price_signal = "HIGH"
        else:
            price_signal = "MID"

        recommended_charge_mwh = 0.0
        recommended_discharge_mwh = 0.0
        grid_charge_requested = False
        recommended_action = "HOLD"
        decision_reason = "NO_ACTION"

        predicted_surplus = predicted_pv_mwh - schedule_mwh

        if scenario_name == "NO_BATTERY":
            recommended_action = "NO_BATTERY"
            decision_reason = "BASELINE_DIRECT_PV_EXPORT"

        elif scenario_name == "BATTERY_BALANCING_PV_ONLY":
            if predicted_surplus > 0:
                recommended_charge_mwh = predicted_surplus
                recommended_action = "CHARGE"
                decision_reason = "PREDICTED_PV_SURPLUS"
            else:
                recommended_discharge_mwh = -predicted_surplus
                recommended_action = "DISCHARGE"
                decision_reason = "PREDICTED_PV_DEFICIT"

        elif scenario_name == "BATTERY_PRICE_GRID":
            if price_signal == "LOW":
                recommended_charge_mwh = MAX_CHARGE_PER_INTERVAL_MWH
                grid_charge_requested = True
                recommended_action = "CHARGE"
                decision_reason = "LOW_PREDICTED_PRICE"
            elif price_signal == "HIGH":
                recommended_discharge_mwh = MAX_DISCHARGE_PER_INTERVAL_MWH
                recommended_action = "DISCHARGE"
                decision_reason = "HIGH_PREDICTED_PRICE"
            elif predicted_surplus > 0:
                recommended_charge_mwh = predicted_surplus
                recommended_action = "CHARGE"
                decision_reason = "MID_PRICE_PV_SURPLUS"
            else:
                recommended_discharge_mwh = -predicted_surplus
                recommended_action = "DISCHARGE"
                decision_reason = "MID_PRICE_PV_DEFICIT"

        maximum_charge_from_soc = max(
            (BATTERY_MAX_SOC_MWH - soc_mwh) / CHARGE_EFFICIENCY,
            0.0,
        )
        maximum_discharge_from_soc = max(
            (soc_mwh - BATTERY_MIN_SOC_MWH) * DISCHARGE_EFFICIENCY,
            0.0,
        )

        if grid_charge_requested:
            available_charge_source_mwh = (
                actual_pv_mwh + MAX_IMPORT_PER_INTERVAL_MWH
            )
        else:
            available_charge_source_mwh = max(
                actual_pv_mwh - schedule_mwh,
                0.0,
            )

        executed_charge_mwh = min(
            recommended_charge_mwh,
            MAX_CHARGE_PER_INTERVAL_MWH,
            maximum_charge_from_soc,
            available_charge_source_mwh,
        )

        export_headroom_mwh = max(
            MAX_EXPORT_PER_INTERVAL_MWH - actual_pv_mwh,
            0.0,
        )
        executed_discharge_mwh = min(
            recommended_discharge_mwh,
            MAX_DISCHARGE_PER_INTERVAL_MWH,
            maximum_discharge_from_soc,
            export_headroom_mwh,
        )

        pv_to_battery_mwh = min(
            executed_charge_mwh,
            actual_pv_mwh,
        )
        grid_to_battery_mwh = max(
            executed_charge_mwh - pv_to_battery_mwh,
            0.0,
        )
        battery_to_grid_mwh = executed_discharge_mwh

        pv_available_for_grid_mwh = max(
            actual_pv_mwh - pv_to_battery_mwh,
            0.0,
        )
        pv_to_grid_mwh = min(
            pv_available_for_grid_mwh,
            max(
                MAX_EXPORT_PER_INTERVAL_MWH - battery_to_grid_mwh,
                0.0,
            ),
        )
        curtailed_energy_mwh = max(
            pv_available_for_grid_mwh - pv_to_grid_mwh,
            0.0,
        )

        gross_grid_export_mwh = (
            pv_to_grid_mwh + battery_to_grid_mwh
        )
        net_export_mwh = (
            gross_grid_export_mwh - grid_to_battery_mwh
        )

        soc_before_mwh = soc_mwh
        soc_mwh = (
            soc_mwh
            + executed_charge_mwh * CHARGE_EFFICIENCY
            - executed_discharge_mwh / DISCHARGE_EFFICIENCY
        )
        soc_mwh = min(
            max(soc_mwh, BATTERY_MIN_SOC_MWH),
            BATTERY_MAX_SOC_MWH,
        )

        if grid_to_battery_mwh > 1e-12 and pv_to_battery_mwh > 1e-12:
            energy_action = "CHARGE_FROM_PV_AND_GRID"
            charge_source = "PV_AND_GRID"
        elif grid_to_battery_mwh > 1e-12:
            energy_action = "CHARGE_FROM_GRID"
            charge_source = "GRID"
        elif pv_to_battery_mwh > 1e-12:
            energy_action = "CHARGE_FROM_PV"
            charge_source = "PV"
        elif battery_to_grid_mwh > 1e-12:
            energy_action = "BATTERY_TO_GRID"
            charge_source = "NONE"
        elif curtailed_energy_mwh > 1e-12:
            energy_action = "CURTAILMENT"
            charge_source = "NONE"
        elif pv_to_grid_mwh > 1e-12:
            energy_action = "PV_TO_GRID"
            charge_source = "NONE"
        else:
            energy_action = "HOLD"
            charge_source = "NONE"

        if executed_charge_mwh > 1e-12:
            action = "CHARGE"
        elif executed_discharge_mwh > 1e-12:
            action = "DISCHARGE"
        else:
            action = "HOLD"

        dam_revenue_usd = schedule_mwh * dam_price
        realtime_adjustment_usd = (
            net_export_mwh - schedule_mwh
        ) * actual_fmm_price
        degradation_cost_usd = (
            executed_discharge_mwh * DEGRADATION_COST_USD_MWH
        )
        net_revenue_usd = (
            dam_revenue_usd
            + realtime_adjustment_usd
            - degradation_cost_usd
        )

        rows.append({
            "scenario": scenario_name,
            "decision_id": row["decision_id"],
            "decision_time_utc": row["decision_time_utc"],
            "target_interval_start_utc": row["target_interval_start_utc"],
            "target_interval_end_utc": row["target_interval_end_utc"],
            "prediction_source": row["prediction_source"],
            "recommended_action": recommended_action,
            "action": action,
            "energy_action": energy_action,
            "decision_reason": decision_reason,
            "price_signal": price_signal,
            "charge_source": charge_source,
            "predicted_price_usd_mwh": predicted_price,
            "actual_fmm_price_usd_mwh": actual_fmm_price,
            "dam_price_usd_mwh": dam_price,
            "dam_schedule_mwh": schedule_mwh,
            "predicted_pv_energy_mwh": predicted_pv_mwh,
            "actual_pv_energy_mwh": actual_pv_mwh,
            "pv_to_grid_mwh": pv_to_grid_mwh,
            "pv_to_battery_mwh": pv_to_battery_mwh,
            "grid_to_battery_mwh": grid_to_battery_mwh,
            "battery_to_grid_mwh": battery_to_grid_mwh,
            "gross_grid_export_mwh": gross_grid_export_mwh,
            "battery_charge_mwh": executed_charge_mwh,
            "battery_discharge_mwh": executed_discharge_mwh,
            "soc_before_mwh": soc_before_mwh,
            "soc_after_mwh": soc_mwh,
            "net_export_mwh": net_export_mwh,
            "schedule_deviation_mwh": net_export_mwh - schedule_mwh,
            "curtailed_energy_mwh": curtailed_energy_mwh,
            "dam_revenue_usd": dam_revenue_usd,
            "realtime_adjustment_usd": realtime_adjustment_usd,
            "degradation_cost_usd": degradation_cost_usd,
            "net_revenue_usd": net_revenue_usd,
        })

    return pd.DataFrame(rows)


simulation_results_pdf = pd.concat(
    [simulate_scenario(scenario) for scenario in SCENARIOS],
    ignore_index=True,
)

quality_results_pdf = (
    simulation_results_pdf
    .groupby("scenario")
    .agg(
        row_count=("decision_id", "count"),
        minimum_soc=("soc_after_mwh", "min"),
        maximum_soc=("soc_after_mwh", "max"),
        maximum_charge=("battery_charge_mwh", "max"),
        maximum_discharge=("battery_discharge_mwh", "max"),
        minimum_net_export=("net_export_mwh", "min"),
        maximum_net_export=("net_export_mwh", "max"),
    )
    .reset_index()
)

display(quality_results_pdf)

###4) Porównanie finansowe

In [0]:
TERMINAL_PRICE_USD_MWH = float(
    backtest_pdf
    .iloc[-1]["actual_fmm_lmp_usd_mwh"]
)

summary_rows = []

for scenario_name in SCENARIOS:
    scenario_pdf = (
        simulation_results_pdf[
            simulation_results_pdf["scenario"]
            == scenario_name
        ]
        .sort_values(
            "target_interval_start_utc"
        )
    )

    final_soc_mwh = float(
        scenario_pdf.iloc[-1]["soc_after_mwh"]
    )

    if scenario_name == "NO_BATTERY":
        terminal_soc_adjustment_usd = 0.0

    elif final_soc_mwh < BATTERY_INITIAL_SOC_MWH:
        missing_soc_mwh = (
            BATTERY_INITIAL_SOC_MWH
            - final_soc_mwh
        )

        terminal_soc_adjustment_usd = -(
            missing_soc_mwh
            / CHARGE_EFFICIENCY
            * TERMINAL_PRICE_USD_MWH
        )

    else:
        additional_soc_mwh = (
            final_soc_mwh
            - BATTERY_INITIAL_SOC_MWH
        )

        terminal_soc_adjustment_usd = (
            additional_soc_mwh
            * DISCHARGE_EFFICIENCY
            * TERMINAL_PRICE_USD_MWH
        )

    raw_net_revenue_usd = float(
        scenario_pdf["net_revenue_usd"].sum()
    )

    adjusted_net_revenue_usd = (
        raw_net_revenue_usd
        + terminal_soc_adjustment_usd
    )

    total_export_mwh = float(
        scenario_pdf["gross_grid_export_mwh"].sum()
    )

    total_grid_import_mwh = float(
        scenario_pdf["grid_to_battery_mwh"].sum()
    )

    summary_rows.append({
        "scenario": scenario_name,
        "raw_net_revenue_usd":
            raw_net_revenue_usd,
        "terminal_soc_adjustment_usd":
            terminal_soc_adjustment_usd,
        "adjusted_net_revenue_usd":
            adjusted_net_revenue_usd,
        "total_export_mwh":
            total_export_mwh,
        "total_grid_import_mwh":
            total_grid_import_mwh,
        "pv_to_grid_mwh": float(
            scenario_pdf["pv_to_grid_mwh"].sum()
        ),
        "pv_to_battery_mwh": float(
            scenario_pdf["pv_to_battery_mwh"].sum()
        ),
        "grid_to_battery_mwh": float(
            scenario_pdf["grid_to_battery_mwh"].sum()
        ),
        "battery_to_grid_mwh": float(
            scenario_pdf["battery_to_grid_mwh"].sum()
        ),
        "curtailed_energy_mwh": float(
            scenario_pdf["curtailed_energy_mwh"].sum()
        ),
        "battery_charge_mwh":
            float(
                scenario_pdf[
                    "battery_charge_mwh"
                ].sum()
            ),
        "battery_discharge_mwh":
            float(
                scenario_pdf[
                    "battery_discharge_mwh"
                ].sum()
            ),
        "equivalent_cycles":
            float(
                scenario_pdf[
                    "battery_discharge_mwh"
                ].sum()
                /
                (
                    BATTERY_MAX_SOC_MWH
                    - BATTERY_MIN_SOC_MWH
                )
            ),
        "absolute_schedule_deviation_mwh":
            float(
                (
                    scenario_pdf["net_export_mwh"]
                    - scenario_pdf[
                        "dam_schedule_mwh"
                    ]
                ).abs().sum()
            ),
        "final_soc_mwh": final_soc_mwh,
    })

financial_summary_pdf = pd.DataFrame(
    summary_rows
)

baseline_revenue = float(
    financial_summary_pdf.loc[
        financial_summary_pdf["scenario"]
        == "NO_BATTERY",
        "adjusted_net_revenue_usd"
    ].iloc[0]
)

financial_summary_pdf[
    "revenue_difference_vs_no_battery_usd"
] = (
    financial_summary_pdf[
        "adjusted_net_revenue_usd"
    ]
    - baseline_revenue
)

display(financial_summary_pdf)

5

In [0]:
BACKTEST_TABLE = (
    "workspace.energy_gold."
    "battery_dispatch_backtest_15m"
)

BACKTEST_SUMMARY_TABLE = (
    "workspace.energy_gold."
    "battery_backtest_summary"
)

backtest_output_df = (
    spark.createDataFrame(
        simulation_results_pdf
    )
    .withColumn(
        "_simulation_version",
        F.lit("battery_rule_based_explicit_flows_v2")
    )
    .withColumn(
        "_simulation_created_at_utc",
        F.current_timestamp()
    )
)

summary_output_df = (
    spark.createDataFrame(
        financial_summary_pdf
    )
    .withColumn(
        "battery_power_mw",
        F.lit(BATTERY_POWER_MW)
    )
    .withColumn(
        "battery_capacity_mwh",
        F.lit(BATTERY_CAPACITY_MWH)
    )
    .withColumn(
        "round_trip_efficiency",
        F.lit(ROUND_TRIP_EFFICIENCY)
    )
    .withColumn(
        "degradation_cost_usd_mwh",
        F.lit(DEGRADATION_COST_USD_MWH)
    )
    .withColumn(
        "low_price_threshold",
        F.lit(LOW_PRICE_THRESHOLD)
    )
    .withColumn(
        "high_price_threshold",
        F.lit(HIGH_PRICE_THRESHOLD)
    )
    .withColumn(
        "_simulation_created_at_utc",
        F.current_timestamp()
    )
)

(
    backtest_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BACKTEST_TABLE)
)

(
    summary_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BACKTEST_SUMMARY_TABLE)
)

print(f"Backtest: {spark.table(BACKTEST_TABLE).count()}")
print(
    f"Podsumowanie: "
    f"{spark.table(BACKTEST_SUMMARY_TABLE).count()}"
)

###6) Końcowe testy jakości backtestu

In [0]:
saved_backtest_df = spark.table(
    "workspace.energy_gold."
    "battery_dispatch_backtest_15m"
)

duplicate_count = (
    saved_backtest_df
    .groupBy("scenario", "decision_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

soc_violation_count = (
    saved_backtest_df
    .filter(
        (
            F.col("soc_after_mwh")
            < BATTERY_MIN_SOC_MWH - 1e-9
        )
        |
        (
            F.col("soc_after_mwh")
            > BATTERY_MAX_SOC_MWH + 1e-9
        )
    )
    .count()
)

simultaneous_action_count = (
    saved_backtest_df
    .filter(
        (F.col("battery_charge_mwh") > 0)
        &
        (F.col("battery_discharge_mwh") > 0)
    )
    .count()
)

battery_power_violation_count = (
    saved_backtest_df
    .filter(
        (
            F.col("battery_charge_mwh")
            > MAX_CHARGE_PER_INTERVAL_MWH + 1e-9
        )
        |
        (
            F.col("battery_discharge_mwh")
            > MAX_DISCHARGE_PER_INTERVAL_MWH + 1e-9
        )
    )
    .count()
)

poi_violation_count = (
    saved_backtest_df
    .filter(
        (
            F.col("net_export_mwh")
            > MAX_EXPORT_PER_INTERVAL_MWH + 1e-9
        )
        |
        (
            F.col("net_export_mwh")
            < -MAX_IMPORT_PER_INTERVAL_MWH - 1e-9
        )
    )
    .count()
)

energy_flow_error_count = (
    saved_backtest_df
    .filter(
        (
            F.abs(
                F.col("actual_pv_energy_mwh")
                - (
                    F.col("pv_to_grid_mwh")
                    + F.col("pv_to_battery_mwh")
                    + F.col("curtailed_energy_mwh")
                )
            ) > 1e-8
        )
        |
        (
            F.abs(
                F.col("battery_charge_mwh")
                - (
                    F.col("pv_to_battery_mwh")
                    + F.col("grid_to_battery_mwh")
                )
            ) > 1e-8
        )
        |
        (
            F.abs(
                F.col("battery_discharge_mwh")
                - F.col("battery_to_grid_mwh")
            ) > 1e-8
        )
        |
        (
            F.abs(
                F.col("net_export_mwh")
                - (
                    F.col("pv_to_grid_mwh")
                    + F.col("battery_to_grid_mwh")
                    - F.col("grid_to_battery_mwh")
                )
            ) > 1e-8
        )
    )
    .count()
)

revenue_error_count = (
    saved_backtest_df
    .filter(
        F.abs(
            F.col("net_revenue_usd")
            - (
                F.col("dam_revenue_usd")
                + F.col("realtime_adjustment_usd")
                - F.col("degradation_cost_usd")
            )
        ) > 1e-6
    )
    .count()
)

scenario_counts_df = (
    saved_backtest_df
    .groupBy("scenario")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_count"
        )
    )
)

test_results = {
    "duplicate_count": duplicate_count,
    "soc_violation_count": soc_violation_count,
    "simultaneous_action_count":
        simultaneous_action_count,
    "battery_power_violation_count":
        battery_power_violation_count,
    "poi_violation_count": poi_violation_count,
    "energy_flow_error_count": energy_flow_error_count,
    "revenue_error_count": revenue_error_count,
}

display(scenario_counts_df)

for test_name, test_value in test_results.items():
    print(f"{test_name}: {test_value}")

if any(test_results.values()):
    raise RuntimeError(
        "Testy jakości backtestu nie zostały zaliczone."
    )

print("Wszystkie testy backtestu zostały zaliczone.")